# Lesson 10 - AI Agents wey dem dey use for Production

For this lesson you go learn **production patterns** for AI agents wey dey use Microsoft Agent Framework (Python). We go cover:

- **Observability** — dey add timing and logging to how agents dey interact
- **Evaluation** — dey use evaluator agent to score di quality of responses
- **Cost management** — ways to save tokens and choose model wey go beta

Di scenario na **travel agent** wey dey help users plan trips, with monitoring and evaluation dey on top.


## How to set am up


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Wetin to Consider for Production

To move AI agents from prototypes go production, you gats pay careful attention to three pillars:

1. **Observability** — You need make you fit see wetin the agent dey do, how long e dey take, and which tools e dey call. If no tracing and logging, e near impossible to debug production wahala.

2. **Evaluation** — Automated quality checks dey make sure say the agent's responses still dey accurate, complete, and helpful as time dey go. One evaluator agent fit score responses based on the criteria wey dem don define.

3. **Cost Management** — How you use tokens dey directly affect cost. Strategies like prompt optimization, model selection, and caching dey help keep expenses under control without sacrificing quality.


## How to build an agent wey you fit observe

We dey define travel tools an wrap di agent call wit timing so we fit monitor di latency. For production you go integrate wit OpenTelemetry or another similar tracing backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Patterns wey dem dey use for evaluation

One common production pattern na to use another agent as an **evaluator**. Di evaluator go score di primary agent's response against predefined criteria like whether e complete, correct, and helpful.

Dis dey enable:
- Automated quality gates wey go run before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of how agent dey perform over time


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategies wey fit help manage cost

To dey control cost na important for production AI agents. Below na di main strategies:

| Way | Explanation |
|---|---|
| **Prompt optimization** | Make system instructions short and clear. Comot redundant context so input tokens go reduce. |
| **Model selection** | Use small, cheap models (e.g. GPT-4o-mini) for simple tasks like classification or extraction, and keep big models only for complex reasoning. |
| **Caching** | Store tool results and frequent queries make you no dey call the API again for the same thing. |
| **Token budgets** | Set `max_tokens` limits so responses no go unexpectedly long. |
| **Batching** | Group many user queries into one API call where you fit. |

For real, one tiered approach dey work well: route straightforward requests to a fast, inexpensive model and escalate only complex queries to a more capable one.


## Wetin We Learn

For dis lesson you don learn how to:

1. **Make interactions dey observable** for agents with timing and logging, so e go lay ground for tracing and monitoring.
2. **Check agent responses** automatically with one evaluator agent wey go give dem score for completeness, accuracy, and helpfulness.
3. **Manage cost dem** by optimizing prompts, choosing models well, caching, and setting token budgets.

These production patterns help ensure your AI agents are reliable, measurable, and cost-effective at scale.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimah:
Dis document don translate by AI translation service (Co-op Translator: https://github.com/Azure/co-op-translator). Even though we dey try make am accurate, abeg note say automatic translations fit get errors or mistakes. Di original document for im own language suppose be di official source. If na serious or important information, e better make professional human translator handle am. We no dey liable for any misunderstanding or wrong interpretation wey fit come from using this translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
